In [2]:
from groq import Groq
print("✓ Groq ready")

✓ Groq ready


In [ ]:
import sys
import os

# Groq API key
GROQ_API_KEY = ""  
GROQ_MODEL = "llama-3.1-8b-instant"

# Project paths
BASE_DIR = os.path.expanduser("~/Final project")
DATA_DIR = os.path.join(BASE_DIR, "data")
VECTOR_DB_DIR = os.path.join(BASE_DIR, "vector_db")
BM25_DIR = os.path.join(BASE_DIR, "bm25_index")

# Create folders
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(VECTOR_DB_DIR, exist_ok=True)
os.makedirs(BM25_DIR, exist_ok=True)

print("✓ Folders created")
print("✓ Base dir:", BASE_DIR)
print("✓ Data dir:", DATA_DIR)

✓ Folders created
✓ Base dir: /users/PGS0411/pdhanish/Final project
✓ Data dir: /users/PGS0411/pdhanish/Final project/data


In [4]:
from datasets import load_dataset

print("Downloading dataset...")
dataset = load_dataset("enelpol/rag-mini-bioasq", "question-answer-passages", split="test")

print(f"✓ Dataset loaded: {len(dataset)} records")
print("Columns:", dataset.column_names)
print("Sample:", dataset[0])

/users/PGS0411/pdhanish/.local/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Dataset loaded: 707 records
Columns: ['question', 'answer', 'id', 'relevant_passage_ids']
Sample: {'question': 'Is capmatinib effective for glioblastoma?', 'answer': 'No. Combination of capmatinib buparlisib resulted in no clear activity in patients with recurrent PTEN-deficient glioblastoma.', 'id': 4213, 'relevant_passage_ids': [31776899]}


In [5]:
import pandas as pd
import os

# Convert to dataframe
df = pd.DataFrame({
    'question': [item['question'] for item in dataset],
    'answer': [item['answer'] for item in dataset]
})

# Save to CSV
csv_path = os.path.join(DATA_DIR, "medical_qa.csv")
df.to_csv(csv_path, index=False)

print(f"✓ Saved {len(df)} Q&A pairs to CSV")
print(df.head(3))

✓ Saved 707 Q&A pairs to CSV
                                            question  \
0          Is capmatinib effective for glioblastoma?   
1    Describe the mechanism of action of ibalizumab.   
2  What is the function of Neu5Gc (N-Glycolylneur...   

                                              answer  
0  No. Combination of capmatinib buparlisib resul...  
1  Ibalizumab is a humanized monoclonal antibody ...  
2  N-glycolylneuraminic acid (Neu5Gc) is an immun...  


In [6]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Combine question and answer into single text
docs = []
for _, row in df.iterrows():
    text = f"Question: {row['question']}\nAnswer: {row['answer']}"
    docs.append(text)

# Split into chunks
# Cited: RecursiveCharacterTextSplitter from LangChain docs
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len
)

chunks = text_splitter.create_documents(docs)

print(f"✓ Created {len(chunks)} chunks from {len(docs)} Q&A pairs")
print("\nSample chunk:")
print(chunks[0].page_content)

✓ Created 898 chunks from 707 Q&A pairs

Sample chunk:
Question: Is capmatinib effective for glioblastoma?
Answer: No. Combination of capmatinib buparlisib resulted in no clear activity in patients with recurrent PTEN-deficient glioblastoma.


In [7]:
import chromadb
from chromadb.utils import embedding_functions

print("Creating vector database...")

# Use ChromaDB's built-in embedding - no external dependencies needed
chroma_client = chromadb.PersistentClient(path=VECTOR_DB_DIR)

# Built-in sentence transformer - handles everything internally
ef = embedding_functions.DefaultEmbeddingFunction()

collection = chroma_client.get_or_create_collection(
    name="medical_qa",
    embedding_function=ef
)

# Add documents in batches
texts = [chunk.page_content for chunk in chunks]
ids = [f"doc_{i}" for i in range(len(texts))]

batch_size = 100
for i in range(0, len(texts), batch_size):
    collection.add(
        documents=texts[i:i+batch_size],
        ids=ids[i:i+batch_size]
    )
    print(f"✓ Added batch {i//batch_size + 1}")

print(f"✓ Vector database created with {collection.count()} documents")

Creating vector database...


2026-05-07 00:27:40.734432551 [E:onnxruntime:Default, env.cc:234 ThreadMain] pthread_setaffinity_np failed for thread: 931751, index: 5, mask: {16, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-05-07 00:27:40.743463204 [E:onnxruntime:Default, env.cc:234 ThreadMain] pthread_setaffinity_np failed for thread: 931752, index: 6, mask: {18, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-05-07 00:27:40.748442750 [E:onnxruntime:Default, env.cc:234 ThreadMain] pthread_setaffinity_np failed for thread: 931753, index: 7, mask: {14, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-05-07 00:27:40.753436672 [E:onnxruntime:Default, env.cc:234 ThreadMain] pthread_setaffinity_np failed for thread: 931754, index: 8, mask: {10, }, error code: 22 error msg: Invalid argument. Specify the n

✓ Added batch 1


2026-05-07 00:29:32.020456920 [E:onnxruntime:Default, env.cc:234 ThreadMain] pthread_setaffinity_np failed for thread: 932735, index: 6, mask: {18, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-05-07 00:29:32.029447509 [E:onnxruntime:Default, env.cc:234 ThreadMain] pthread_setaffinity_np failed for thread: 932736, index: 7, mask: {14, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-05-07 00:29:32.034427209 [E:onnxruntime:Default, env.cc:234 ThreadMain] pthread_setaffinity_np failed for thread: 932737, index: 8, mask: {10, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-05-07 00:29:32.039425596 [E:onnxruntime:Default, env.cc:234 ThreadMain] pthread_setaffinity_np failed for thread: 932738, index: 9, mask: {20, }, error code: 22 error msg: Invalid argument. Specify the n

✓ Added batch 2


...

✓ Added batch 7


2026-05-07 00:40:43.570236982 [E:onnxruntime:Default, env.cc:234 ThreadMain] pthread_setaffinity_np failed for thread: 934551, index: 6, mask: {18, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-05-07 00:40:43.578446371 [E:onnxruntime:Default, env.cc:234 ThreadMain] pthread_setaffinity_np failed for thread: 934552, index: 7, mask: {14, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-05-07 00:40:43.583426648 [E:onnxruntime:Default, env.cc:234 ThreadMain] pthread_setaffinity_np failed for thread: 934553, index: 8, mask: {10, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-05-07 00:40:43.589426962 [E:onnxruntime:Default, env.cc:234 ThreadMain] pthread_setaffinity_np failed for thread: 934554, index: 9, mask: {20, }, error code: 22 error msg: Invalid argument. Specify the n

✓ Added batch 8


2026-05-07 00:42:35.220034294 [E:onnxruntime:Default, env.cc:234 ThreadMain] pthread_setaffinity_np failed for thread: 934795, index: 6, mask: {18, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-05-07 00:42:35.228444922 [E:onnxruntime:Default, env.cc:234 ThreadMain] pthread_setaffinity_np failed for thread: 934796, index: 7, mask: {14, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-05-07 00:42:35.233429324 [E:onnxruntime:Default, env.cc:234 ThreadMain] pthread_setaffinity_np failed for thread: 934797, index: 8, mask: {10, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-05-07 00:42:35.238425523 [E:onnxruntime:Default, env.cc:234 ThreadMain] pthread_setaffinity_np failed for thread: 934798, index: 9, mask: {20, }, error code: 22 error msg: Invalid argument. Specify the n

✓ Added batch 9
✓ Vector database created with 898 documents


In [8]:
from rank_bm25 import BM25Okapi
import pickle
import os

# Prepare corpus
texts = [chunk.page_content for chunk in chunks]
tokenized_corpus = [text.lower().split() for text in texts]

# Build BM25 index
# Citation: Robertson & Zaragoza, 2009 - BM25 ranking function
bm25 = BM25Okapi(tokenized_corpus)

# Save index
bm25_path = os.path.join(BM25_DIR, "bm25_index.pkl")
with open(bm25_path, "wb") as f:
    pickle.dump((bm25, texts), f)

print(f"✓ BM25 index built with {len(texts)} documents")
print(f"✓ Saved to {bm25_path}")


✓ BM25 index built with 898 documents
✓ Saved to /users/PGS0411/pdhanish/Final project/bm25_index/bm25_index.pkl


In [9]:
import pickle

# Load BM25
bm25_path = os.path.join(BM25_DIR, "bm25_index.pkl")
with open(bm25_path, "rb") as f:
    bm25, texts = pickle.load(f)

def hybrid_search(query, n_results=5):
    # BM25 keyword search
    tokenized_query = query.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)
    top_bm25_idx = sorted(range(len(bm25_scores)), 
                          key=lambda i: bm25_scores[i], reverse=True)[:n_results]
    bm25_results = [texts[i] for i in top_bm25_idx]
    
    # Semantic search from ChromaDB
    semantic_results = collection.query(
        query_texts=[query],
        n_results=n_results
    )
    semantic_docs = semantic_results['documents'][0]
    semantic_score = 1 - semantic_results['distances'][0][0]
    
    # Combine results (remove duplicates)
    combined = list(dict.fromkeys(bm25_results + semantic_docs))[:n_results]
    
    return combined, round(semantic_score * 100, 1)

# Test
results, confidence = hybrid_search("What is paracetamol?")
print(f"✓ Hybrid search working")
print(f"✓ Confidence score: {confidence}%")
print(f"✓ Retrieved {len(results)} documents")
print("\nTop result:")
print(results[0][:200])

2026-05-07 00:44:23.589867928 [E:onnxruntime:Default, env.cc:234 ThreadMain] pthread_setaffinity_np failed for thread: 934977, index: 6, mask: {18, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-05-07 00:44:23.598444902 [E:onnxruntime:Default, env.cc:234 ThreadMain] pthread_setaffinity_np failed for thread: 934978, index: 7, mask: {14, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-05-07 00:44:23.603425962 [E:onnxruntime:Default, env.cc:234 ThreadMain] pthread_setaffinity_np failed for thread: 934979, index: 8, mask: {10, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-05-07 00:44:23.607965523 [E:onnxruntime:Default, env.cc:234 ThreadMain] pthread_setaffinity_np failed for thread: 934980, index: 9, mask: {20, }, error code: 22 error msg: Invalid argument. Specify the n

✓ Hybrid search working
✓ Confidence score: -27.9%
✓ Retrieved 5 documents

Top result:
Question: What is MPE-seq?


In [10]:
def hybrid_search(query, n_results=5):
    # BM25 keyword search
    tokenized_query = query.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)
    top_bm25_idx = sorted(range(len(bm25_scores)), 
                          key=lambda i: bm25_scores[i], reverse=True)[:n_results]
    bm25_results = [texts[i] for i in top_bm25_idx]
    
    # Semantic search from ChromaDB
    semantic_results = collection.query(
        query_texts=[query],
        n_results=n_results
    )
    semantic_docs = semantic_results['documents'][0]
    
    # Fix confidence: convert distance to similarity percentage
    distance = semantic_results['distances'][0][0]
    confidence = round(max(0, (2 - distance) / 2 * 100), 1)
    
    # Combine results (remove duplicates)
    combined = list(dict.fromkeys(bm25_results + semantic_docs))[:n_results]
    
    return combined, confidence

# Test
results, confidence = hybrid_search("What is the treatment for diabetes?")
print(f"✓ Hybrid search working")
print(f"✓ Confidence score: {confidence}%")
print(f"✓ Retrieved {len(results)} documents")
print("\nTop result:")
print(results[0][:200])

2026-05-07 00:44:29.580539935 [E:onnxruntime:Default, env.cc:234 ThreadMain] pthread_setaffinity_np failed for thread: 935019, index: 6, mask: {18, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-05-07 00:44:29.589444426 [E:onnxruntime:Default, env.cc:234 ThreadMain] pthread_setaffinity_np failed for thread: 935020, index: 7, mask: {14, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-05-07 00:44:29.594430197 [E:onnxruntime:Default, env.cc:234 ThreadMain] pthread_setaffinity_np failed for thread: 935021, index: 8, mask: {10, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-05-07 00:44:29.599426308 [E:onnxruntime:Default, env.cc:234 ThreadMain] pthread_setaffinity_np failed for thread: 935022, index: 9, mask: {20, }, error code: 22 error msg: Invalid argument. Specify the n

✓ Hybrid search working
✓ Confidence score: 44.3%
✓ Retrieved 5 documents

Top result:
Question: What disease is the drug aducanumab targeting?
Answer: Aducanumab is an anti-Aβ antibody being developed for the treatment of Alzheimer's disease (AD).


In [12]:
from groq import Groq
import json
client = Groq(api_key=GROQ_API_KEY)

# Safety layer — keywords and urgency thresholds loaded from LLM
def load_emergency_keywords():
    import json
    prompt = """You are a medical triage expert.
List medical emergency keywords indicating life-threatening situations requiring immediate 911 response.
Return ONLY a valid JSON array of short phrases, nothing else.
Example: ["chest pain", "difficulty breathing"]"""
    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=200
    )
    text = response.choices[0].message.content.strip()
    start = text.find("[")
    end = text.rfind("]") + 1
    if start == -1 or end == 0:
        return []
    return json.loads(text[start:end])

EMERGENCY_KEYWORDS = load_emergency_keywords()
print(f"✓ Loaded {len(EMERGENCY_KEYWORDS)} emergency keywords from LLM")
print(f"  Keywords: {EMERGENCY_KEYWORDS}")

def check_safety(query):
    query_lower = query.lower()
    for keyword in EMERGENCY_KEYWORDS:
        if keyword in query_lower:
            return True, keyword
    return False, None

def get_urgency_level(query, confidence):
    import json
    prompt = f"""You are a medical triage assistant.
A patient asked: "{query}"
Retrieval confidence score: {confidence}%

Classify into exactly one urgency level:
- HIGH: emergency symptoms requiring immediate 911 response
- MODERATE: symptoms that may worsen and need doctor visit
- LOW: mild symptoms manageable at home
- INFO: research or informational question with no symptoms

Return ONLY a JSON object:
{{"urgency": "HIGH/MODERATE/LOW/INFO", "reason": "one short sentence"}}"""

    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=100
    )
    text = response.choices[0].message.content.strip()
    start = text.find("{")
    end = text.rfind("}") + 1
    result = json.loads(text[start:end])
    return result["urgency"], result["reason"]

# Test
is_emergency, keyword = check_safety("I have chest pain and difficulty breathing")
print(f"✓ Safety layer working")
print(f"✓ Emergency detected: {is_emergency}, keyword: {keyword}")

level, reason = get_urgency_level("I have a mild headache", 65)
print(f"✓ Urgency level: {level} - {reason}")

✓ Loaded 10 emergency keywords from LLM
  Keywords: ['severe allergic reaction', 'chest pain', 'difficulty breathing', 'uncontrolled bleeding', 'severe head injury', 'seizure', 'stroke symptoms', 'heart attack', 'loss of consciousness', 'abdominal pain']
✓ Safety layer working
✓ Emergency detected: True, keyword: chest pain
✓ Urgency level: LOW - Typical headache symptoms can usually be managed with over-the-counter pain relief.


In [13]:
from groq import Groq
client = Groq(api_key=GROQ_API_KEY)

def get_medical_response(query, patient_profile=None):
    # Step 1: Safety check via LLM urgency
    retrieved_docs, confidence = hybrid_search(query, n_results=5)
    urgency, urgency_reason = get_urgency_level(query, confidence)

    if urgency == "HIGH":
        return {
            "response": "EMERGENCY: Please call 911 or go to the nearest emergency room immediately.",
            "confidence": 0,
            "urgency": "HIGH",
            "urgency_reason": urgency_reason,
            "sources": []
        }

    # Step 2: Build personalized prompt dynamically
    context = "\n\n".join(retrieved_docs[:3])

    profile_text = ""
    if patient_profile:
        profile_text = f"""
Patient Profile:
- Name: {patient_profile.get('name', 'Unknown')}
- Age: {patient_profile.get('age', 'Unknown')}
- Gender: {patient_profile.get('gender', 'Unknown')}
- Known conditions: {patient_profile.get('conditions', 'None')}
"""

    prompt = f"""You are MediQ, an AI medical assistant talking directly to the patient.
Always use "you" and "your" — never say "the patient".
Answer in 2-3 short sentences. No greetings. No disclaimers.
If it is a research or scientific question, just explain what it is.

{profile_text}
Medical Context:
{context}

Patient Question: {query}
Response:"""

    # Step 3: Generate response
    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=300
    )

    return {
        "response": response.choices[0].message.content,
        "confidence": confidence,
        "urgency": urgency,
        "urgency_reason": urgency_reason,
        "sources": retrieved_docs[:2]
    }

# Test
profile = {"name": "John", "age": "24", "gender": "Male", "conditions": "None"}
result = get_medical_response("What are the symptoms of diabetes?", profile)
print(f"✓ Response generated")
print(f"✓ Confidence: {result['confidence']}%")
print(f"✓ Urgency: {result['urgency']}")
print(f"\nResponse:\n{result['response']}")

2026-05-07 00:48:24.181676880 [E:onnxruntime:Default, env.cc:234 ThreadMain] pthread_setaffinity_np failed for thread: 935544, index: 6, mask: {18, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-05-07 00:48:24.190455366 [E:onnxruntime:Default, env.cc:234 ThreadMain] pthread_setaffinity_np failed for thread: 935545, index: 7, mask: {14, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-05-07 00:48:24.195426781 [E:onnxruntime:Default, env.cc:234 ThreadMain] pthread_setaffinity_np failed for thread: 935546, index: 8, mask: {10, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-05-07 00:48:24.200427466 [E:onnxruntime:Default, env.cc:234 ThreadMain] pthread_setaffinity_np failed for thread: 935547, index: 9, mask: {20, }, error code: 22 error msg: Invalid argument. Specify the n

✓ Response generated
✓ Confidence: 41.0%
✓ Urgency: INFO

Response:
Diabetes typically includes symptoms like increased thirst, frequent urination, fatigue, blurred vision, slow healing cuts or injuries, and tingling or numbness in your hands and feet. Some people might not experience these symptoms at first, especially if their blood sugar levels aren't extremely high. As a result, type 2 diabetes often goes undiagnosed for a long time.
